# Project Ouranos: terrain segmentation training (Kaggle)

Run on Kaggle with a GPU and the dataset **yash92328/ai4mars-terrainaware-autonomous-driving-on-mars** attached
(Add Input > Datasets). Outputs in `/kaggle/working/`:

- `terrain_unet.keras`: trained model, copy to `models/`
- `metrics.json`: test results for U-Net and baseline, copy to `models/`
- `demo_frames.zip`: test frames and expert labels for the dashboard, unzip into `demo_frames/`

The U-Net here is the same architecture as `src/model.py`.

In [ ]:
import glob, json, os, random, shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from PIL import Image

IMG_SIZE = 256
NUM_CLASSES = 4
CLASS_NAMES = ["soil", "bedrock", "sand", "big rock"]
NULL_LABEL = 255
MAX_TRAIN = 8000      # images used for training (full set is ~16k; cap for time)
EPOCHS = 10
BATCH = 16
random.seed(0); np.random.seed(0); tf.random.set_seed(0)
print("GPU:", tf.config.list_physical_devices("GPU"))

In [ ]:
# Locate the dataset folders (robust to how Kaggle nests the files)
edr_dirs = glob.glob("/kaggle/input/**/msl/images/edr", recursive=True)
train_dirs = glob.glob("/kaggle/input/**/msl/labels/train", recursive=True)
test_dirs = glob.glob("/kaggle/input/**/msl/labels/test/masked-gold-min3-100agree", recursive=True)
print(edr_dirs, train_dirs, test_dirs, sep="\n")
EDR, TRAIN_LABELS, TEST_LABELS = edr_dirs[0], train_dirs[0], test_dirs[0]

def pairs(label_dir):
    out = []
    for lab in sorted(glob.glob(os.path.join(label_dir, "*.png"))):
        stem = os.path.basename(lab).replace("_merged", "").replace(".png", "")
        img = os.path.join(EDR, stem + ".JPG")
        if os.path.exists(img):
            out.append((img, lab))
    return out

train_pairs = pairs(TRAIN_LABELS)
test_pairs = pairs(TEST_LABELS)
random.shuffle(train_pairs)
train_pairs = train_pairs[:MAX_TRAIN]
n_val = int(0.1 * len(train_pairs))
val_pairs, train_pairs = train_pairs[:n_val], train_pairs[n_val:]
print("train", len(train_pairs), "val", len(val_pairs), "test", len(test_pairs))

In [ ]:
def load_pair(img_path, lab_path):
    img = Image.open(img_path).convert("L").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR)
    lab = Image.open(lab_path).resize((IMG_SIZE, IMG_SIZE), Image.NEAREST)
    lab = np.asarray(lab)
    if lab.ndim == 3:
        lab = lab[:, :, 0]
    return np.asarray(img, dtype=np.float32) / 255.0, lab.astype(np.uint8)

def load_all(pairs_list):
    X = np.zeros((len(pairs_list), IMG_SIZE, IMG_SIZE, 1), dtype=np.float32)
    Y = np.zeros((len(pairs_list), IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    for i, (a, b) in enumerate(pairs_list):
        X[i, :, :, 0], Y[i] = load_pair(a, b)
    return X, Y

X_train, Y_train = load_all(train_pairs)
X_val, Y_val = load_all(val_pairs)
X_test, Y_test = load_all(test_pairs)

# Class frequencies in training labels (ignoring unlabelled pixels)
counts = np.array([(Y_train == c).sum() for c in range(NUM_CLASSES)], dtype=np.float64)
print(dict(zip(CLASS_NAMES, (counts / counts.sum()).round(4))))

# Inverse-frequency class weights so rare hazards (sand, big rock) matter in training
class_w = (counts.sum() / (NUM_CLASSES * counts))
class_w = np.clip(class_w, 0.5, 10.0)
print("class weights", dict(zip(CLASS_NAMES, class_w.round(2))))

def weights_and_targets(Y):
    w = np.zeros(Y.shape, dtype=np.float32)
    for c in range(NUM_CLASSES):
        w[Y == c] = class_w[c]
    target = Y.copy()
    target[Y == NULL_LABEL] = 0   # value doesn't matter, weight is 0
    return target, w

T_train, W_train = weights_and_targets(Y_train)
T_val, W_val = weights_and_targets(Y_val)

In [ ]:
def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
    x = layers.BatchNormalization()(x)
    return x

def build_unet(img_size=IMG_SIZE, num_classes=NUM_CLASSES, base=16):
    inputs = layers.Input((img_size, img_size, 1))
    c1 = conv_block(inputs, base); p1 = layers.MaxPooling2D()(c1)
    c2 = conv_block(p1, base * 2); p2 = layers.MaxPooling2D()(c2)
    c3 = conv_block(p2, base * 4); p3 = layers.MaxPooling2D()(c3)
    b = conv_block(p3, base * 8)
    u3 = conv_block(layers.Concatenate()([layers.UpSampling2D()(b), c3]), base * 4)
    u2 = conv_block(layers.Concatenate()([layers.UpSampling2D()(u3), c2]), base * 2)
    u1 = conv_block(layers.Concatenate()([layers.UpSampling2D()(u2), c1]), base)
    outputs = layers.Conv2D(num_classes, 1, activation="softmax")(u1)
    return tf.keras.Model(inputs, outputs, name="terrain_unet")

model = build_unet()
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy")
history = model.fit(
    X_train, T_train, sample_weight=W_train,
    validation_data=(X_val, T_val, W_val),
    epochs=EPOCHS, batch_size=BATCH,
    callbacks=[tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)],
)

In [ ]:
def confusion(pred, Y):
    valid = Y != NULL_LABEL
    cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=np.int64)
    np.add.at(cm, (Y[valid].astype(int), pred[valid].astype(int)), 1)
    return cm

def scores(cm, prefix):
    out = {}
    ious = []
    for c, name in enumerate(CLASS_NAMES):
        tp = cm[c, c]; fp = cm[:, c].sum() - tp; fn = cm[c, :].sum() - tp
        iou = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0.0
        out[prefix + " IoU " + name] = float(iou)
        ious.append(iou)
    out[prefix + " mean IoU"] = float(np.mean(ious))
    out[prefix + " pixel accuracy"] = float(np.trace(cm) / cm.sum())
    return out

pred_test = model.predict(X_test, batch_size=BATCH).argmax(axis=-1)
unet_cm = confusion(pred_test, Y_test)
metrics = scores(unet_cm, "U-Net")
for k, v in metrics.items(): print(k, round(v, 3))

In [ ]:
# Baseline: per-pixel logistic regression on brightness and local texture (no deep learning)
from scipy.ndimage import uniform_filter
from sklearn.linear_model import LogisticRegression

def pixel_features(X):
    img = X[..., 0]
    feats = [img]
    for size in (5, 15):
        mean = uniform_filter(img, size=(1, size, size))
        sq = uniform_filter(img ** 2, size=(1, size, size))
        feats += [mean, np.sqrt(np.clip(sq - mean ** 2, 0, None))]
    return np.stack(feats, axis=-1)

F_train = pixel_features(X_train[:500])
Yb = Y_train[:500]
valid = Yb != NULL_LABEL
idx = np.random.choice(valid.sum(), size=min(200000, valid.sum()), replace=False)
clf = LogisticRegression(max_iter=300, class_weight="balanced")
clf.fit(F_train[valid][idx], Yb[valid][idx])

F_test = pixel_features(X_test)
pred_base = clf.predict(F_test.reshape(-1, F_test.shape[-1])).reshape(Y_test.shape)
metrics.update(scores(confusion(pred_base, Y_test), "Baseline"))
for k, v in metrics.items(): print(k, round(v, 3))

In [ ]:
# Export model, metrics and demo frames
OUT = "/kaggle/working"
model.save(os.path.join(OUT, "terrain_unet.keras"))
metrics["test images"] = len(test_pairs)
metrics["train images"] = len(train_pairs)
with open(os.path.join(OUT, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

# Demo frames: a mix of test frames with sand, with big rocks, and mostly safe terrain
def frac(Y, c):
    valid = Y != NULL_LABEL
    return (Y[valid] == c).mean() if valid.any() else 0.0

order = list(range(len(test_pairs)))
sandy = sorted(order, key=lambda i: -frac(Y_test[i], 2))[:10]
rocky = sorted([i for i in order if i not in sandy], key=lambda i: -frac(Y_test[i], 3))[:8]
rest = [i for i in order if i not in sandy and i not in rocky]
safe = random.sample(rest, 12)
chosen = safe[:4] + sandy[:3] + safe[4:8] + rocky[:3] + sandy[3:6] + safe[8:] + rocky[3:6] + sandy[6:]

demo_dir = os.path.join(OUT, "demo_frames")
os.makedirs(demo_dir, exist_ok=True)
for n, i in enumerate(chosen):
    img_path, lab_path = test_pairs[i]
    name = "frame_" + str(n).zfill(2)
    Image.open(img_path).convert("L").resize((512, 512)).save(os.path.join(demo_dir, name + ".jpg"), quality=90)
    Image.open(lab_path).resize((512, 512), Image.NEAREST).save(os.path.join(demo_dir, name + "_label.png"))
shutil.make_archive(os.path.join(OUT, "demo_frames"), "zip", demo_dir)
print("saved", len(chosen), "demo frames")